In [1]:
!pip install sacrebleu rouge nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 784.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 2.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import sacrebleu
from rouge import Rouge
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
import nltk
from google.colab import drive


# Ensure you have the necessary NLTK data
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [3]:
drive.mount('/content/drive/')
%cd "/content/drive/My Drive/PhD/RE24/OnlineAppendix/RQ2_Results/Requirements Lists"

Mounted at /content/drive/
/content/drive/My Drive/PhD/RE24/OnlineAppendix/RQ2_Results/Requirements Lists


In [4]:
llm = []
experts = []
recover = []

with open('recover_requirements.txt') as f:
    recover = [line.rstrip() for line in f]

with open('llm_only_requirements.txt') as f:
    llm = [line.rstrip() for line in f]

with open('experts_final_requirements.txt') as f:
    experts = [line.rstrip() for line in f]

In [5]:
print("RECOVER'S requirements: #", len(recover))
print("LLM generated requirements: #", len(llm))
print("Experts' requirements: #", len(experts))

RECOVER'S requirements: # 135
LLM generated requirements: # 31
Experts' requirements: # 28


In [11]:
# Tokenize the sentences
tokenized_recover = [nltk.word_tokenize(sent) for sent in recover]
tokenized_llm = [nltk.word_tokenize(sent) for sent in llm]
tokenized_experts = [nltk.word_tokenize(sent) for sent in experts]

In [68]:
def compute_corpus_scores(reference_sentences, hypothesis_sentences):
    # BLEU score for the entire corpus
    bleu = sacrebleu.corpus_bleu([" ".join(hyp) for hyp in hypothesis_sentences], [[" ".join(ref)] for ref in reference_sentences])
    bleu_score = bleu.score
    bp = bleu.bp
    ratio = bleu.sys_len / bleu.ref_len
    print("BP: ", bp)
    print("Lenght Ratio: ", ratio)

    # ROUGE score for the entire corpus
    rouge = Rouge()
    joined_ref = " ".join([" ".join(ref) for ref in reference_sentences])
    joined_hyp = " ".join([" ".join(hyp) for hyp in hypothesis_sentences])
    rouge_score = rouge.get_scores(joined_hyp, joined_ref)[0]

    joined_ref = nltk.word_tokenize(joined_ref)
    joined_hyp = nltk.word_tokenize(joined_hyp)
    # METEOR score for the entire corpus by averaging sentence-level METEOR scores
    meteor_scores = [meteor_score([joined_ref], joined_hyp)]
    avg_meteor_score = np.mean(meteor_scores)

    return bleu_score, rouge_score['rouge-1']['f']*100, avg_meteor_score*100

# Experts vs LLM

In [69]:
# Compute scores
bleu_score, rouge_score, avg_meteor_score = compute_corpus_scores(tokenized_experts, tokenized_llm)



# Display results
results = pd.DataFrame({
    'Metric': ['BLEU', 'ROUGE-1', 'METEOR'],
    'Score': [bleu_score, rouge_score,  avg_meteor_score]
})

print(results)

BP:  0.7613003866968737
Lenght Ratio:  0.7857142857142857
    Metric      Score
0     BLEU  49.363593
1  ROUGE-1  43.961352
2   METEOR  35.344286


# Experts vs RECOVER

In [70]:
# Compute scores
bleu_score, rouge_score, avg_meteor_score = compute_corpus_scores(tokenized_experts, tokenized_recover)



# Display results
results = pd.DataFrame({
    'Metric': ['BLEU', 'ROUGE-1', 'METEOR'],
    'Score': [bleu_score, rouge_score,  avg_meteor_score]
})

print(results)

BP:  0.925961078642316
Lenght Ratio:  0.9285714285714286
    Metric      Score
0     BLEU  78.819297
1  ROUGE-1  37.195122
2   METEOR  39.098515


# LLM vs RECOVER


In [71]:
# Compute scores
bleu_score, rouge_score, avg_meteor_score = compute_corpus_scores(tokenized_llm, tokenized_recover)



# Display results
results = pd.DataFrame({
    'Metric': ['BLEU', 'ROUGE-1', 'METEOR'],
    'Score': [bleu_score, rouge_score,  avg_meteor_score]
})

print(results)

BP:  1.0
Lenght Ratio:  1.0
    Metric      Score
0     BLEU  57.835699
1  ROUGE-1  37.837837
2   METEOR  32.764922


# EXPERT BY EXPERT ANALYSIS


In [59]:
expert1 = []
expert2 = []
expert3 = []

with open('expert_1_requirements.txt') as f:
    expert1 = [line.rstrip() for line in f]

with open('expert_2_requirements.txt') as f:
    expert2 = [line.rstrip() for line in f]

with open('expert_3_requirements.txt') as f:
    expert3 = [line.rstrip() for line in f]

In [60]:
# Tokenize the sentences
tokenized_1 = [nltk.word_tokenize(sent) for sent in expert1]
tokenized_2 = [nltk.word_tokenize(sent) for sent in expert2]
tokenized_3 = [nltk.word_tokenize(sent) for sent in expert3]

## Expert 1 vs Recover

In [72]:
# Compute scores
bleu_score, rouge_score, avg_meteor_score = compute_corpus_scores(tokenized_1, tokenized_recover)

# Display results
results = pd.DataFrame({
    'Metric': ['BLEU', 'ROUGE-1', 'METEOR'],
    'Score': [bleu_score, rouge_score,  avg_meteor_score]
})

print(results)

BP:  1.0
Lenght Ratio:  1.0
    Metric      Score
0     BLEU  37.239099
1  ROUGE-1  48.179871
2   METEOR  39.927251


## Expert 2 vs RECOVER

In [73]:
# Compute scores
bleu_score, rouge_score, avg_meteor_score = compute_corpus_scores(tokenized_2, tokenized_recover)

# Display results
results = pd.DataFrame({
    'Metric': ['BLEU', 'ROUGE-1', 'METEOR'],
    'Score': [bleu_score, rouge_score,  avg_meteor_score]
})

print(results)

BP:  1.0
Lenght Ratio:  1.0
    Metric      Score
0     BLEU  92.347326
1  ROUGE-1  28.616852
2   METEOR  40.226924


## Expert 3 vs RECOVER

In [74]:
# Compute scores
bleu_score, rouge_score, avg_meteor_score = compute_corpus_scores(tokenized_3, tokenized_recover)

# Display results
results = pd.DataFrame({
    'Metric': ['BLEU', 'ROUGE-1', 'METEOR'],
    'Score': [bleu_score, rouge_score,  avg_meteor_score]
})

print(results)

BP:  0.925961078642316
Lenght Ratio:  0.9285714285714286
    Metric      Score
0     BLEU  30.115406
1  ROUGE-1  26.130653
2   METEOR  28.570312
